# Análise dos resultados de segmentação e detecção dos óstios

## Objetivo

Este notebook realiza uma análise exploratória dos resultados da segmentação dos óstios a partir do pipeline. São apresentados gráficos e estatísticas para avaliar a qualidade das segmentações.

Resume status dos óstios, distâncias e Dice dos resultados canônicos de treino, validação e teste.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (
    NOTEBOOK_CWD,
    NOTEBOOK_CWD.parent,
    NOTEBOOK_CWD.parent.parent,
):
    src_dir = candidate / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

from utils.project.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment()


In [ ]:
import pandas as pd

from utils.project.notebook_env import get_default_split_paths
from utils.comparison_utils import filter_correct_ostia_cases, load_split_summary
from utils.visualization.comparison import (
    plot_dice_distribution_by_subset,
    plot_dice_distribution_for_publication,
)
from utils.visualization.segmentation_eda import (
    build_success_status_summary_by_subset,
    plot_distance_distribution_by_subset,
    plot_status_distribution_by_subset,
    plot_success_error_by_subset,
)

SUCCESS_STATUS = ["ambos toleráveis", "ambos corretos"]
BAD_OSTIA_STATUS = ["óstios não encontrados", "óstios ruins (bloqueado)"]


## Configuração

Ajuste nesta seção apenas os parâmetros da análise; o pipeline base não é alterado.

## Carregamento dos Dados
Os dados de saída do pipeline são carregados a partir do arquivo CSV gerado, contendo as informações de status e coordenadas dos óstios para cada imagem.

In [ ]:
SPLIT_PATHS_BY_RESOLUTION = get_default_split_paths(REPO_ROOT)
VALID_SPLITS = ("train", "val", "test")
RESOLUTION_KEYS = {"high": "high_res", "mid": "mid_res"}


def load_summary_df(resolution: str, split: str) -> pd.DataFrame | None:
    return load_split_summary(
        SPLIT_PATHS_BY_RESOLUTION,
        RESOLUTION_KEYS[resolution],
        split,
    )


# Carregar todos os dados
data = {
    resolution: {split: load_summary_df(resolution, split) for split in VALID_SPLITS}
    for resolution in RESOLUTION_KEYS
}

# Dataset atualmente ativo (padrão: mid/train)
current_resolution = 'mid'
current_split = 'train'
df = data[current_resolution][current_split]

available_pairs = [
    f"{resolution}/{split}"
    for resolution, split_data in data.items()
    for split, split_df in split_data.items()
    if split_df is not None
]
print("Dados carregados com sucesso!")
print("\nResolução e split disponíveis:", ", ".join(available_pairs))
print(f"Dados ativos: {current_resolution}/{current_split}")
df.head()


## Análise

As subseções abaixo apresentam as métricas, tabelas ou visualizações do objetivo definido.

## Distribuição dos Status de Segmentação

### Treino (Train)

In [ ]:
plot_status_distribution_by_subset(data, 'train')

### Validação (Val)

In [ ]:
plot_status_distribution_by_subset(data, 'val')

### Teste (Test)

In [ ]:
plot_status_distribution_by_subset(data, 'test')

## Acertos e Erros na Segmentação

### Treino (Train)

In [ ]:
plot_success_error_by_subset(data, 'train', SUCCESS_STATUS)
build_success_status_summary_by_subset(data, 'train', SUCCESS_STATUS)

### Validação (Val)

In [ ]:
plot_success_error_by_subset(data, 'val', SUCCESS_STATUS)
build_success_status_summary_by_subset(data, 'val', SUCCESS_STATUS)

### Teste (Test)

In [ ]:
plot_success_error_by_subset(data, 'test', SUCCESS_STATUS)
build_success_status_summary_by_subset(data, 'test', SUCCESS_STATUS)

## Distribuição das Distâncias Left e Right (mm)

A linha tracejada vermelha em **7 mm** representa o limite de tolerância adotado para considerar a detecção do óstio como aceitável.

### Treino (Train)

In [ ]:
plot_distance_distribution_by_subset(data, 'train', bins=25)

### Validação (Val)

In [ ]:
plot_distance_distribution_by_subset(data, 'val', bins=15)

### Teste (Test)

In [ ]:
plot_distance_distribution_by_subset(data, 'test', bins=25)

## Distribuição do Dice Score

### Treino (Train)

In [ ]:
plot_dice_distribution_by_subset(data['mid']['train'], data['high']['train'], "Treino - Todos os casos")

In [ ]:
plot_dice_distribution_by_subset(
    filter_correct_ostia_cases(data['mid']['train']),
    filter_correct_ostia_cases(data['high']['train']),
    "Treino - Sem óstios incorretos",
)

### Validação (Val)

In [ ]:
plot_dice_distribution_by_subset(data['mid']['val'], data['high']['val'], "Validação - Todos os casos")

In [ ]:
plot_dice_distribution_by_subset(
    filter_correct_ostia_cases(data['mid']['val']),
    filter_correct_ostia_cases(data['high']['val']),
    "Validação - Sem óstios incorretos",
)

### Teste (Test)

In [ ]:
plot_dice_distribution_by_subset(data['mid']['test'], data['high']['test'], "Teste - Todos os casos")

In [ ]:
plot_dice_distribution_by_subset(
    filter_correct_ostia_cases(data['mid']['test']),
    filter_correct_ostia_cases(data['high']['test']),
    "Teste - Sem óstios incorretos",
)

## Figuras de Dice em mid resolution para publicação
As celulas abaixo geram figuras em alta resolucao separando os casos com sucesso de ostios e os casos com ostios incorretos.


In [ ]:
TEMP_FIGURE_DPI = 400
TEMP_DICE_SPLIT = "test"
TEMP_FIGURE_DIR = REPO_ROOT / "output/segmentation/analysis/segmentation_results/figures"
TEMP_FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def build_mid_dice_subset(
    only_correct_ostia: bool,
    split: str = TEMP_DICE_SPLIT,
) -> pd.DataFrame:
    subset = data["mid"][split].copy()
    if only_correct_ostia:
        subset = filter_correct_ostia_cases(subset)
    subset["dice_artery"] = pd.to_numeric(subset["dice_artery"], errors="coerce")
    return subset.dropna(subset=["dice_artery"])


In [ ]:
mid_dice_all_cases_df = build_mid_dice_subset(only_correct_ostia=False)
_, all_cases_stats = plot_dice_distribution_for_publication(
    mid_dice_all_cases_df["dice_artery"],
    output_path=TEMP_FIGURE_DIR / "mid_resolution_test_dice_all_cases.png",
    dpi=TEMP_FIGURE_DPI,
)
print(all_cases_stats)


In [ ]:
mid_dice_correct_ostia_df = build_mid_dice_subset(only_correct_ostia=True)
_, correct_ostia_stats = plot_dice_distribution_for_publication(
    mid_dice_correct_ostia_df["dice_artery"],
    output_path=TEMP_FIGURE_DIR / "mid_resolution_test_dice_correct_ostia.png",
    dpi=TEMP_FIGURE_DPI,
)
print(correct_ostia_stats)


## Conclusão

Os resumos de status, distância e Dice permitem avaliar os splits disponíveis e separar o impacto da detecção dos óstios sobre a segmentação arterial.